# Video Restoration Master 2026 (v4)

**Cambios de la v4 sobre la v3** (dos auditorías más, una de rendimiento y otra de arquitectura — ambas contrastadas contra el código fuente real antes de implementar nada):

- **Corrección importante**: `--chunk_size`, `--video_backend ffmpeg` y `--use_10bit` **sí existen** en el wrapper que este notebook usa (`numz/ComfyUI-SeedVR2_VideoUpscaler`) — se confirmó leyendo `inference_cli.py` directamente. La v3 los rechazaba por error. Ahora se activan automáticamente si el `--help` instalado los reporta.
- VFR: en vez de fingir preservarlo (el wrapper internamente usa OpenCV y colapsa a un FPS único, así que no hay timestamps reales que preservar), se **normaliza explícitamente a CFR** antes de procesar, y se avisa cuando esto pasa.
- Detección de cortes de plano antes de restaurar, para no mezclar contexto temporal entre planos distintos (usa `--cache_dit`/`--cache_vae` del propio CLI para procesar por carpeta sin recargar el modelo).
- Batch size calculado con la VRAM libre real (`torch.cuda.mem_get_info`), no solo por tabla fija de GPU.
- Selector explícito de variante de modelo: `3b` / `7b` / `7b_sharp` / `auto` (auto sigue siendo conservador por defecto).
- Pre-limpieza específica según el tipo de degradación medido (bloques -> deband, blur ya alto -> denoise más suave), no una receta única.
- Métricas de calidad ahora muestrean por tiempo real, no por índice de frame (falla en VFR).
- Diagnóstico temporal corre a 480p en vez de full-res (mismo resultado, mucho más rápido).
- Procesos largos ya no bloquean el notebook en silencio: se transmiten a un log en vivo.
- Chequeo de cuota de disco antes de escribir intermedios pesados.
- `color_range` y valores `unknown`/`unspecified` de ffprobe ahora se manejan bien (antes solo se detectaba `None`).
- Entrega H.264 usa NVENC (hardware) si está disponible, con fallback a libx264.
- Real-ESRGAN ahora prueba sus propios flags contra `--help` antes de usarlos, en vez de asumirlos — hubo inconsistencias entre copias del script encontradas en la web y no quiero adivinar y volver a romper la celda.

**Lo que se decidió NO implementar, y por qué:**
- Integrar FlashVSR/PS-SR/DGAF-VSR/InstantViR/STCDiT como motores adicionales de benchmark — son hallazgos de investigación reales, pero convertirlo en un banco de pruebas de 5 modelos es un proyecto aparte.
- Seguir ampliando el clasificador de calidad con más señales (noise/motion/saturation/chroma) — ya se mejoró dos veces, sigue siendo heurístico por diseño, y los retornos son decrecientes.
- Reconstrucción exacta de timestamps VFR frame por frame (Ruta A) — el wrapper no preserva esa información internamente de todas formas; normalizar a CFR explícito es la solución honesta con el motor que este notebook usa.

### Antes de empezar
`Entorno de ejecución → Cambiar tipo de entorno de ejecución`: **A100** o **L4**. Ejecuta las celdas **en orden**.

## 0. Verificar GPU asignada

In [ ]:
import subprocess, torch
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True).stdout.strip()
print('GPU detectada:', gpu_info if gpu_info else 'NINGUNA — ve a Entorno de ejecución > Cambiar tipo de entorno y elige una GPU.')
GPU_NAME = gpu_info.split(',')[0].strip() if gpu_info else ''
print('Nombre GPU:', GPU_NAME)
print('CUDA disponible para torch:', torch.cuda.is_available())

## 1a. Instalar Real-ESRGAN v0.3.0
Se guarda el `--help` real del script en `REALESRGAN_HELP` para no asumir flags que puedan no existir en esta copia exacta.

In [ ]:
import os, subprocess, glob

REALESRGAN_DIR = '/content/Real-ESRGAN'
if os.path.isdir(REALESRGAN_DIR):
    subprocess.run(['rm', '-rf', REALESRGAN_DIR], check=True)

subprocess.run(['git', 'clone', '-q', '--branch', 'v0.3.0', '--depth', '1',
                 'https://github.com/xinntao/Real-ESRGAN.git', REALESRGAN_DIR], check=True)

subprocess.run(['pip', 'install', '-q', 'numpy<2'], check=True)
subprocess.run(['pip', 'install', '-q', 'basicsr', 'facexlib', 'gfpgan'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REALESRGAN_DIR, check=True)
subprocess.run(['python', 'setup.py', 'develop', '-q'], cwd=REALESRGAN_DIR, check=True)

targets = (glob.glob('/usr/**/basicsr/**/*.py', recursive=True)
           + glob.glob('/usr/**/facexlib/**/*.py', recursive=True)
           + glob.glob('/usr/**/gfpgan/**/*.py', recursive=True))
patched = 0
for f in targets:
    try:
        content = open(f, encoding='utf-8').read()
    except Exception:
        continue
    if 'torchvision.transforms.functional_tensor' in content:
        open(f, 'w', encoding='utf-8').write(
            content.replace('torchvision.transforms.functional_tensor', 'torchvision.transforms.functional'))
        patched += 1
print(f'Parche aplicado a {patched} archivo(s).')

weights_dir = os.path.join(REALESRGAN_DIR, 'weights')
os.makedirs(weights_dir, exist_ok=True)
subprocess.run(['wget', '-q', '-nc', '-P', weights_dir,
                 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'], check=True)

import importlib
import basicsr.data.degradations
importlib.reload(basicsr.data.degradations)

help_res = subprocess.run(['python', 'inference_realesrgan_video.py', '--help'],
                           cwd=REALESRGAN_DIR, capture_output=True, text=True)
REALESRGAN_HELP = help_res.stdout + help_res.stderr
print(REALESRGAN_HELP[:1500])
print('\n✅ Real-ESRGAN v0.3.0 listo.')

## 1b. Instalar SeedVR2
Se fuerza `numpy<2` una segunda vez después de instalar requirements (por si algún paquete transitivo lo sube a 2.x), y se guarda el `--help` completo en `SEEDVR2_HELP` para activar `--chunk_size`/`--video_backend`/`--use_10bit` solo si de verdad existen en esta copia.

In [ ]:
import os, subprocess

SEEDVR2_DIR = '/content/seedvr2_videoupscaler'
if os.path.isdir(SEEDVR2_DIR):
    subprocess.run(['rm', '-rf', SEEDVR2_DIR], check=True)

subprocess.run(['git', 'clone', '-q', 'https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git', SEEDVR2_DIR], check=True)

req_path = os.path.join(SEEDVR2_DIR, 'requirements.txt')
req_colab_path = os.path.join(SEEDVR2_DIR, 'requirements_colab.txt')
with open(req_path) as f:
    lines = f.readlines()
filtered = [l for l in lines if not l.strip().lower().startswith(('torch', 'torchvision', 'torchaudio', 'numpy'))]
filtered.append('numpy<2\n')
with open(req_colab_path, 'w') as f:
    f.writelines(filtered)

subprocess.run(['pip', 'install', '-q', '-r', req_colab_path], check=True)
# Guardia final: por si algún paquete transitivo volvió a subir numpy a 2.x
subprocess.run(['pip', 'install', '-q', 'numpy<2', '--force-reinstall', '--no-deps'], check=True)

import torch
print('Torch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
import numpy
print('NumPy:', numpy.__version__)

help_res = subprocess.run(['python', 'inference_cli.py', '--help'], cwd=SEEDVR2_DIR,
                           capture_output=True, text=True)
SEEDVR2_HELP = help_res.stdout + help_res.stderr
print(SEEDVR2_HELP[:1500])
print('...')
for flag in ('--chunk_size', '--video_backend', '--use_10bit', '--10bit', '--cache_dit', '--cache_vae'):
    print(f'{flag}: {"disponible" if flag in SEEDVR2_HELP else "NO disponible en esta copia"}')
print('\n✅ SeedVR2 CLI listo.')

## 2. Sube tu video

In [ ]:
from google.colab import files
import os, shutil, re

os.chdir('/content')
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No se subió ningún archivo.')

raw_name = next(iter(uploaded.keys()))
safe_name = os.path.basename(raw_name)
safe_name = re.sub(r'[^A-Za-z0-9._-]', '_', safe_name)
if not safe_name.lower().endswith(('.mp4', '.mov', '.mkv', '.avi', '.webm')):
    raise ValueError(f'Formato de video no soportado: {safe_name}')

input_filename = safe_name
input_path = os.path.join('/content/input', input_filename)
shutil.move(raw_name, input_path)
print(f'Video subido: {input_path}')

## 3. Análisis del video (ffprobe)
El bitrate ahora prioriza el canal de video (`vstream`) sobre el bitrate global del contenedor (que mezcla audio) — mezclarlos podía desplazar la clasificación CLEAN/NORMAL/SEVERE hasta un 20-50% en videos con audio de alta tasa. La metadata de color ahora también reconoce `unknown`/`unspecified` como "no declarado", no solo `None`.

In [ ]:
import subprocess, json as _json

def ffprobe_json(path):
    cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format', '-show_streams', path]
    res = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return _json.loads(res.stdout)

probe = ffprobe_json(input_path)
vstreams = [s for s in probe['streams'] if s['codec_type'] == 'video']
if not vstreams:
    raise ValueError('El archivo no contiene una pista de video.')
vstream = vstreams[0]
astreams = [s for s in probe['streams'] if s['codec_type'] == 'audio']
fmt = probe['format']

width = int(vstream['width'])
height = int(vstream['height'])

def parse_rate(s, default=25.0):
    try:
        num, den = s.split('/')
        den = float(den)
        return float(num) / den if den else default
    except Exception:
        return default

r_fps = parse_rate(vstream.get('r_frame_rate', '25/1'))
avg_fps = parse_rate(vstream.get('avg_frame_rate', '25/1'))
fps = r_fps
IS_VFR = abs(r_fps - avg_fps) > 0.05

duration = float(fmt.get('duration', vstream.get('duration', 0)) or 0)
nb_frames = int(vstream.get('nb_frames', 0) or (duration * fps))

# Prioriza el bitrate del canal de video; format.bit_rate mezcla audio+video+overhead
bitrate = int(vstream.get('bit_rate', 0) or fmt.get('bit_rate', 0) or 0)

codec = vstream.get('codec_name', 'desconocido')
pix_fmt = vstream.get('pix_fmt', 'desconocido')
has_audio = len(astreams) > 0

INVALID_COLOR = {None, '', 'unknown', 'unspecified', 'reserved'}
def clean_color(v, default):
    return v if v not in INVALID_COLOR else default

color_space = clean_color(vstream.get('color_space'), 'bt709')
color_primaries = clean_color(vstream.get('color_primaries'), 'bt709')
color_transfer = clean_color(vstream.get('color_transfer'), 'bt709')
color_range = clean_color(vstream.get('color_range'), 'tv')
color_declared = vstream.get('color_space') not in INVALID_COLOR

print('--- Info del video ---')
print(f'Resolución:     {width}x{height}')
print(f'FPS (r/avg):    {r_fps:.3f} / {avg_fps:.3f}  {"(VFR detectado)" if IS_VFR else "(CFR)"}')
print(f'Duración:       {duration:.1f} s  |  Frames (aprox): {nb_frames}')
print(f'Códec:          {codec}  |  pix_fmt: {pix_fmt}')
print(f'Color:          space={color_space} primaries={color_primaries} transfer={color_transfer} range={color_range}'
      + ('  (declarado)' if color_declared else '  (no declarado/desconocido, se asume bt709/tv)'))
print(f'Bitrate (video): {bitrate/1000:.0f} kbps' if bitrate else 'Bitrate:        no reportado, se estimará')
print(f'Audio:          {"sí" if has_audio else "no"} ({len(astreams)} pista(s))')

if not bitrate and duration > 0:
    size_bytes = int(fmt.get('size', 0) or 0)
    bitrate = int(size_bytes * 8 / duration) if size_bytes else 0
    print(f'Bitrate estimado por tamaño de archivo: {bitrate/1000:.0f} kbps (incluye audio, aproximado)')

## 4. Normalización a CFR (si aplica)
El motor SeedVR2 lee el video con OpenCV internamente y trabaja con un único FPS — no preserva timestamps VFR reales sin importar lo que se haga después con ffmpeg. En vez de fingir que se preserva (como hacía la v3 con `-fps_mode passthrough`), si se detecta VFR se normaliza explícitamente a CFR aquí, una sola vez, y se avisa.

In [ ]:
import subprocess

cfr_path = '/content/work/cfr_normalized.mp4'

if IS_VFR:
    print('VFR detectado. Este pipeline no puede preservar timestamps variables de extremo a extremo')
    print(f'con el motor que usa (SeedVR2 vía OpenCV), así que se normaliza explícitamente a {fps:.3f} fps CFR.')
    cmd = ['ffmpeg', '-y', '-i', input_path, '-r', str(fps),
           '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '0',
           '-c:a', 'copy', cfr_path]
    subprocess.run(cmd, check=True)
    working_source = cfr_path
else:
    working_source = input_path
    print('CFR ya detectado, no se necesita normalizar.')

print('Fuente de trabajo:', working_source)

## 5. Clasificación automática de calidad
El muestreo ahora es por tiempo real (`CAP_PROP_POS_MSEC`), no por índice de frame — en video VFR el índice no corresponde linealmente al tiempo real y podía muestrear frames no representativos.

In [ ]:
import cv2, numpy as np

def sample_frames(path, n=5):
    cap = cv2.VideoCapture(path)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    dur_ms = (total_frames / src_fps) * 1000 if src_fps else 0
    if dur_ms <= 0:
        cap.release()
        return []
    ts = np.linspace(0, max(dur_ms - 40, 0), num=n)
    frames = []
    for t in ts:
        cap.set(cv2.CAP_PROP_POS_MSEC, float(t))
        ok, fr = cap.read()
        if ok:
            frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY))
    cap.release()
    return frames

def blur_score(gray_frames):
    if not gray_frames:
        return None
    return float(np.mean([cv2.Laplacian(g, cv2.CV_64F).var() for g in gray_frames]))

def blockiness_score(gray_frames):
    if not gray_frames:
        return None
    ratios = []
    for g in gray_frames:
        gx = np.abs(np.diff(g.astype(np.float32), axis=1))
        cols = np.arange(gx.shape[1])
        boundary_mask = (cols % 8 == 7)
        boundary_mean = gx[:, boundary_mask].mean() if boundary_mask.any() else 0
        nonboundary_mean = gx[:, ~boundary_mask].mean() if (~boundary_mask).any() else 1e-6
        ratios.append(boundary_mean / max(nonboundary_mean, 1e-6))
    return float(np.mean(ratios))

sampled = sample_frames(working_source, n=5)
blur = blur_score(sampled)
blockiness = blockiness_score(sampled)

bpp = bitrate / (width * height * fps) if (width and height and fps and bitrate) else 0

print(f'Bits por píxel (bpp, solo canal de video): {bpp:.4f}')
print(f'Nitidez (Laplaciano): {blur:.1f}' if blur is not None else 'Nitidez: no calculada')
print(f'Bloques de compresión: {blockiness:.3f}' if blockiness is not None else 'Bloques: no calculado')

if bpp >= 0.12 and min(width, height) >= 480:
    TIER = 'CLEAN'
elif bpp >= 0.04:
    TIER = 'NORMAL'
else:
    TIER = 'SEVERE'

if min(width, height) < 360 and TIER == 'CLEAN':
    TIER = 'NORMAL'

BLUR_THRESHOLD = 60.0
BLOCKINESS_THRESHOLD = 1.3

degraded_by_visual = (blur is not None and blur < BLUR_THRESHOLD) or (blockiness is not None and blockiness > BLOCKINESS_THRESHOLD)
if degraded_by_visual and TIER == 'CLEAN':
    TIER = 'NORMAL'
    print('Ajuste: bitrate sugería CLEAN, pero se ve borroso/con bloques -> bajado a NORMAL.')
elif degraded_by_visual and TIER == 'NORMAL':
    TIER = 'SEVERE'
    print('Ajuste: NORMAL con señales visuales fuertes -> subido a SEVERE.')

print(f'\nClasificación final: {TIER}  (heurística: bpp + nitidez + bloques, sigue sin ser un modelo perceptual entrenado)')
# TIER = 'NORMAL'  # para forzar manualmente

## 6. Detección de cortes de plano
Solo aplica a SeedVR2 (NORMAL/SEVERE) — Real-ESRGAN procesa frame por frame, así que no le afecta el contexto temporal entre planos. Si hay demasiados cortes detectados (más de `MAX_SHOTS`), se desactiva para no fragmentar en exceso ni sumar overhead de recarga.

In [ ]:
import subprocess, re

SCENE_THRESHOLD = 0.4
MAX_SHOTS = 15

def detect_shot_cuts(path, fps_val, threshold=SCENE_THRESHOLD):
    cmd = ['ffmpeg', '-i', path, '-vf', f"select='gt(scene,{threshold})',showinfo", '-f', 'null', '-']
    res = subprocess.run(cmd, capture_output=True, text=True)
    times = [float(m) for m in re.findall(r'pts_time:([\d.]+)', res.stderr)]
    return sorted(set(int(round(t * fps_val)) for t in times))

SHOT_AWARE = False
shot_ranges = [(0, nb_frames)]

if TIER != 'CLEAN':
    cut_frames = detect_shot_cuts(working_source, fps)
    print(f'Cortes de plano detectados: {len(cut_frames)}')
    if 0 < len(cut_frames) < MAX_SHOTS:
        SHOT_AWARE = True
        bounds = [0] + cut_frames + [nb_frames]
        shot_ranges = list(zip(bounds[:-1], bounds[1:]))
        shot_ranges = [(s, e) for s, e in shot_ranges if e > s]
    elif len(cut_frames) >= MAX_SHOTS:
        print(f'Demasiados cortes ({len(cut_frames)} >= {MAX_SHOTS}); se procesa como un solo plano.')
    else:
        print('Sin cortes detectados; se procesa como un solo plano.')
else:
    print('TIER=CLEAN usa Real-ESRGAN, la detección de planos no aplica.')

print(f'Modo por-plano activo: {SHOT_AWARE}  |  planos: {len(shot_ranges)}')

## 7. Pre-limpieza específica por tipo de degradación (solo SEVERE)
En vez de aplicar siempre el mismo `hqdn3d`, ahora reacciona a lo que se midió en el paso 5: bloques de compresión visibles suman `deband`; si el video ya está borroso, se evita el denoise agresivo (solo empeoraría la pérdida de detalle). También revisa la cuota de disco antes de escribir un intermedio casi-lossless.

In [ ]:
import subprocess, shutil

work_input = '/content/work/pre_cleaned.mp4'

FREE_GB = shutil.disk_usage('/content').free / 1e9
bps_lossless_est = 100e6 * (width * height * fps) / (1920 * 1080 * 30)
est_gb = duration * bps_lossless_est / 8 / 1e9
crf_preclean = '0' if FREE_GB > est_gb * 2 + 5 else '12'
if crf_preclean != '0':
    print(f'Disco libre {FREE_GB:.1f} GB, estimado necesario ~{est_gb:.1f} GB -> usando crf {crf_preclean} en vez de lossless.')

if TIER == 'SEVERE':
    vf_parts = []
    if blockiness is not None and blockiness > BLOCKINESS_THRESHOLD:
        vf_parts.append('deband')
        print('Bloques de compresión detectados -> se agrega deband.')
    if blur is not None and blur < BLUR_THRESHOLD:
        vf_parts.append('hqdn3d=0.5:0.5:1:1')
        print('El video ya está borroso -> denoise muy ligero para no perder más detalle real.')
    else:
        vf_parts.append('hqdn3d=1.5:1.5:3:3')
    vf = ','.join(vf_parts)
    cmd = ['ffmpeg', '-y', '-i', working_source, '-vf', vf,
           '-c:v', 'libx264', '-preset', 'veryfast', '-crf', crf_preclean,
           '-c:a', 'copy', work_input]
    subprocess.run(cmd, check=True)
else:
    shutil.copy(working_source, work_input)
    print('TIER != SEVERE, sin pre-limpieza.')

print('Archivo de trabajo:', work_input)

## 8. Restauración / upscaling

`SEEDVR2_MODEL_VARIANT`: `'auto'` (conservador, 3B por defecto), `'3b'`, `'7b'` o `'7b_sharp'` para forzar manualmente y comparar tú mismo — hay reportes de artefactos de bandas verticales en los modelos 7B en algunas configuraciones, así que no se activan automáticamente.

El progreso ahora se transmite en vivo a un log (`/content/work/seedvr2_run_*.log`) en vez de quedar bloqueado en silencio hasta que el proceso termine.

In [ ]:
import math, os, glob, subprocess, sys, shutil, gc, torch

TARGET_HEIGHT = 1080
SEEDVR2_MODEL_VARIANT = 'auto'  # 'auto' | '3b' | '7b' | '7b_sharp'

restored_video_path = None

def gpu_tier(gpu_name):
    g = gpu_name.upper()
    if 'A100' in g or 'H100' in g:
        return 'HIGH'
    if 'L4' in g:
        return 'MID'
    return 'LOW'

tier = gpu_tier(GPU_NAME)

def vram_free_gb():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, _ = torch.cuda.mem_get_info()
        return free_b / 1e9
    return 0.0

free_gb = vram_free_gb()
print(f'VRAM libre detectada: {free_gb:.1f} GB')

def pick_seedvr2_model(tier, variant):
    if variant == '3b':
        model = {'HIGH': 'seedvr2_ema_3b_fp16.safetensors',
                 'MID': 'seedvr2_ema_3b_fp8_e4m3fn.safetensors',
                 'LOW': 'seedvr2_ema_3b-Q8_0.gguf'}[tier]
    elif variant == '7b':
        model = 'seedvr2_ema_7b_fp16.safetensors'
        print('⚠️  SeedVR2 7B: reportes de artefactos de bandas verticales en algunas configuraciones.')
    elif variant == '7b_sharp':
        model = 'seedvr2_ema_7b_sharp_fp16.safetensors'
        print('⚠️  SeedVR2 7B Sharp: más detalle, mayor riesgo de sobre-generar en degradación ligera.')
    else:
        model = {'HIGH': 'seedvr2_ema_3b_fp16.safetensors',
                  'MID': 'seedvr2_ema_3b_fp8_e4m3fn.safetensors',
                  'LOW': 'seedvr2_ema_3b-Q8_0.gguf'}[tier]
    extra = {}
    if tier == 'LOW' or 'gguf' in model.lower() or 'q8' in model.lower():
        extra = {'blocks_to_swap': 24, 'dit_offload_device': 'cpu', 'swap_io_components': True}
    return model, extra

dit_model, extra = pick_seedvr2_model(tier, SEEDVR2_MODEL_VARIANT)

TIER_MAX_BATCH = {'HIGH': 33, 'MID': 21, 'LOW': 9}
PER_FRAME_GB = 1.5   # estimado aproximado a 1080p, no validado formalmente
MODEL_BASE_GB = 3.0 if '3b' in dit_model.lower() else 6.0
HEADROOM_GB = 1.5

def pick_batch_size(n_frames, max_batch_tier, free_gb_val):
    candidates = [1, 5, 9, 13, 17, 21, 25, 33, 41]
    max_vram_batch = max(1, int((free_gb_val - MODEL_BASE_GB - HEADROOM_GB) / PER_FRAME_GB)) if free_gb_val > 0 else max_batch_tier
    cap = min(max_batch_tier, max_vram_batch)
    valid = [c for c in candidates if c <= max(n_frames, 1) and c <= cap]
    return (valid[-1] if valid else 1), cap

TEMPORAL_CFG = {'LOW': {'overlap': 1, 'prepend': 2}, 'MID': {'overlap': 2, 'prepend': 3}, 'HIGH': {'overlap': 3, 'prepend': 4}}
cfg = TEMPORAL_CFG[tier]

def run_seedvr2_cli(input_target, output_target, n_frames_hint, extra_flags):
    batch_size, cap = pick_batch_size(n_frames_hint, TIER_MAX_BATCH[tier], free_gb)
    temporal_overlap = cfg['overlap'] if batch_size >= 9 else 0
    prepend_frames = cfg['prepend']
    print(f'  batch_size={batch_size} (cap VRAM/tier={cap})  overlap={temporal_overlap}  prepend={prepend_frames}')

    cmd = ['python', 'inference_cli.py', input_target,
           '--output', output_target,
           '--output_format', 'mp4',
           '--dit_model', dit_model,
           '--resolution', str(TARGET_HEIGHT),
           '--batch_size', str(batch_size),
           '--uniform_batch_size',
           '--temporal_overlap', str(temporal_overlap),
           '--prepend_frames', str(prepend_frames),
           '--color_correction', 'lab']
    if '--video_backend' in SEEDVR2_HELP:
        cmd += ['--video_backend', 'ffmpeg']
    if '--use_10bit' in SEEDVR2_HELP:
        cmd += ['--use_10bit']
    elif '--10bit' in SEEDVR2_HELP:
        cmd += ['--10bit']
    if '--chunk_size' in SEEDVR2_HELP and n_frames_hint > 200:
        cmd += ['--chunk_size', '150']
    cmd += extra_flags

    print('Comando:', ' '.join(cmd))
    log_path = f'/content/work/seedvr2_run_{os.path.basename(str(input_target)).replace("/", "_")}.log'
    proc = subprocess.Popen(cmd, cwd=SEEDVR2_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    KEYWORDS = ('it/s', 's/it', 'Error', 'error', 'Traceback', 'OOM', 'CUDA out of memory', 'Chunk', 'chunk')
    with open(log_path, 'w', buffering=1) as logf:
        for line in proc.stdout:
            logf.write(line)
            if any(k in line for k in KEYWORDS):
                sys.stdout.write(line); sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        tail = open(log_path).read()[-3000:]
        raise RuntimeError(f'SeedVR2 falló (rc={proc.returncode}). Log:\n{tail}\n'
                            'Si el error menciona un flag no reconocido, la CLI cambió de interfaz.')
    return log_path

if TIER == 'CLEAN':
    print('Motor: Real-ESRGAN (realesr-general-x4v3)')
    if height >= TARGET_HEIGHT:
        print(f'Ya tiene {height}px de alto, se omite el upscaling con IA.')
        restored_video_path = work_input
    else:
        outscale = TARGET_HEIGHT / height
        tile = {'LOW': 256, 'MID': 512, 'HIGH': 0}[tier]
        for f in glob.glob('/content/work/*restored*.mp4'):
            os.remove(f)
        cmd = ['python', 'inference_realesrgan_video.py', '-i', work_input,
               '-n', 'realesr-general-x4v3', '-s', '4', '--outscale', f'{outscale:.4f}',
               '-o', '/content/work', '--suffix', 'restored']
        if '-dn' in REALESRGAN_HELP:
            cmd += ['-dn', '0.4']
        else:
            print('Aviso: -dn no aparece en --help, se omite (denoise por defecto del modelo).')
        if '--tile' in REALESRGAN_HELP and tile:
            cmd += ['--tile', str(tile), '--tile_pad', '10']
        res = subprocess.run(cmd, cwd=REALESRGAN_DIR, capture_output=True, text=True)
        if res.returncode != 0:
            print('STDOUT:', res.stdout[-2000:]); print('STDERR:', res.stderr[-2000:])
            raise RuntimeError(f'Real-ESRGAN falló con código {res.returncode}')
        matches = sorted(glob.glob('/content/work/*restored*.mp4'))
        if not matches:
            raise FileNotFoundError('Real-ESRGAN no generó ningún MP4.')
        out = matches[0]
        probe_out = ffprobe_json(out)
        dur_out = float(probe_out['format'].get('duration', 0) or 0)
        if dur_out < 0.95 * duration:
            raise RuntimeError(f'Real-ESRGAN produjo {dur_out:.1f}s, esperado ~{duration:.1f}s (salida truncada).')
        restored_video_path = out
    print('Salida:', restored_video_path)

else:
    print(f'Motor: SeedVR2  |  modelo: {dit_model}  |  GPU tier: {tier}  |  variante: {SEEDVR2_MODEL_VARIANT}')

    extra_flags = []
    if extra.get('blocks_to_swap'):
        extra_flags += ['--blocks_to_swap', str(extra['blocks_to_swap']),
                         '--dit_offload_device', extra.get('dit_offload_device', 'cpu')]
        if extra.get('swap_io_components'):
            extra_flags += ['--swap_io_components']

    if SHOT_AWARE:
        print(f'Modo por-plano: {len(shot_ranges)} planos.')
        shots_dir = '/content/work/shots_in'
        out_dir = '/content/work/shots_out'
        for d in (shots_dir, out_dir):
            if os.path.isdir(d):
                shutil.rmtree(d)
            os.makedirs(d, exist_ok=True)

        shot_files = []
        for i, (start, end) in enumerate(shot_ranges):
            shot_path = os.path.join(shots_dir, f'shot_{i:03d}.mp4')
            t_start = start / fps
            t_dur = max((end - start) / fps, 1 / fps)
            cmd = ['ffmpeg', '-y', '-ss', f'{t_start:.3f}', '-i', work_input, '-t', f'{t_dur:.3f}',
                   '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '0', '-c:a', 'copy', shot_path]
            subprocess.run(cmd, check=True)
            shot_files.append(shot_path)

        if '--cache_dit' in SEEDVR2_HELP:
            extra_flags += ['--cache_dit']
        if '--cache_vae' in SEEDVR2_HELP:
            extra_flags += ['--cache_vae']

        max_shot_len = max(e - s for s, e in shot_ranges)
        run_seedvr2_cli(shots_dir, out_dir, max_shot_len, extra_flags)

        out_matches = sorted(glob.glob(f'{out_dir}/*.mp4'))
        if len(out_matches) != len(shot_files):
            print(f'Aviso: se esperaban {len(shot_files)} planos restaurados, se encontraron {len(out_matches)}.')
        concat_list = '/content/work/concat_list.txt'
        with open(concat_list, 'w') as f:
            for p in out_matches:
                f.write(f"file '{p}'\n")
        restored_video_path = '/content/work/restored_concat.mp4'
        cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list, '-c', 'copy', restored_video_path]
        subprocess.run(cmd, check=True)

    else:
        restored_frames_dir = '/content/work/seedvr2_out'
        if os.path.isdir(restored_frames_dir):
            shutil.rmtree(restored_frames_dir)
        os.makedirs(restored_frames_dir, exist_ok=True)
        run_seedvr2_cli(work_input, restored_frames_dir, nb_frames, extra_flags)
        matches = sorted(glob.glob(f'{restored_frames_dir}/*.mp4'))
        if not matches:
            raise FileNotFoundError(f'SeedVR2 no generó ningún MP4 en {restored_frames_dir}')
        restored_video_path = matches[0]

    probe_out = ffprobe_json(restored_video_path)
    dur_out = float(probe_out['format'].get('duration', 0) or 0)
    if dur_out < 0.9 * duration:
        print(f'Aviso: duración de salida ({dur_out:.1f}s) notablemente menor a la original ({duration:.1f}s). Revisa el resultado.')
    print('Salida:', restored_video_path)

## 9. Diagnóstico de estabilidad temporal (informativo)
Ahora corre a 480p en vez de resolución completa — mismo tipo de resultado, mucho más rápido (el optical flow de OpenCV es single-thread y en CPU).

In [ ]:
import cv2, numpy as np

def temporal_stability_score(path, sample_pairs=25, work_h=480):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None
    n = min(sample_pairs, total - 1)
    idxs = np.linspace(0, total - 2, num=n, dtype=int)
    residuals = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret1, f1 = cap.read()
        ret2, f2 = cap.read()
        if not (ret1 and ret2):
            continue
        g1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
        g2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)
        scale = work_h / g1.shape[0]
        s1 = cv2.resize(g1, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        s2 = cv2.resize(g2, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        flow = cv2.calcOpticalFlowFarneback(s1, s2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        h, w = s1.shape
        grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (grid_x + flow[..., 0]).astype(np.float32)
        map_y = (grid_y + flow[..., 1]).astype(np.float32)
        s2_warped = cv2.remap(s2, map_x, map_y, cv2.INTER_LINEAR)
        residuals.append(np.mean(cv2.absdiff(s1, s2_warped)))
    cap.release()
    return float(np.mean(residuals)) if residuals else None

if restored_video_path:
    score = temporal_stability_score(restored_video_path)
    if score is not None:
        print(f'Residual medio tras compensar movimiento (a 480p): {score:.2f} (escala 0-255)')
        print('Referencia orientativa (no calibrada formalmente): <5 estable, 5-10 normal, >10 revisa visualmente.')
    else:
        print('No se pudo calcular.')
else:
    print('No hay video restaurado para analizar.')

### (Opcional) Aplicar deflicker si el diagnóstico salió alto

In [ ]:
import subprocess

APPLY_DEFLICKER = False

if APPLY_DEFLICKER and restored_video_path:
    deflickered = '/content/work/deflickered.mp4'
    cmd = ['ffmpeg', '-y', '-i', restored_video_path,
           '-vf', 'deflicker=mode=pm',
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '12',
           '-c:a', 'copy', deflickered]
    subprocess.run(cmd, check=True)
    restored_video_path = deflickered
    print('Deflicker aplicado:', restored_video_path)
else:
    print('Deflicker no aplicado.')

## 10. Master 10-bit BT.709 (ProRes 422 HQ)
Ya no necesita decidir entre `-r` y `-fps_mode passthrough`: el video ya quedó normalizado a CFR en el paso 4, así que siempre se usa un FPS constante y explícito aquí.

In [ ]:
import subprocess

master_path = '/content/output/master_1080p_bt709.mov'

cmd = ['ffmpeg', '-y', '-i', restored_video_path]
if has_audio:
    cmd += ['-i', input_path, '-map', '0:v:0', '-map', '1:a:0?', '-c:a', 'pcm_s16le', '-shortest']
else:
    cmd += ['-map', '0:v:0', '-an']

vf = (f"zscale=matrixin={color_space}:matrix=bt709:"
      f"primariesin={color_primaries}:primaries=bt709:"
      f"transferin={color_transfer}:transfer=bt709,format=yuv422p10le")
cmd += ['-vf', vf,
        '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
        '-c:v', 'prores_ks', '-profile:v', '3',
        '-r', str(fps),
        master_path]
subprocess.run(cmd, check=True)

print('Master generado:', master_path)

## 11. Copia de entrega (H.264)
Usa NVENC (encoder de hardware) si está disponible en la GPU asignada, con fallback automático a `libx264` por CPU.

In [ ]:
import subprocess

nvenc_test = subprocess.run(
    ['ffmpeg', '-hide_banner', '-f', 'lavfi', '-i', 'nullsrc=s=64x64:d=0.1',
     '-c:v', 'h264_nvenc', '-f', 'null', '-'],
    capture_output=True)
nvenc_ok = nvenc_test.returncode == 0

delivery_path = '/content/output/delivery_1080p_h264.mp4'
if nvenc_ok:
    print('NVENC disponible, usando encoder de hardware.')
    v_flags = ['-c:v', 'h264_nvenc', '-preset', 'p5', '-cq', '21', '-rc', 'vbr']
else:
    print('NVENC no disponible, usando libx264 (CPU).')
    v_flags = ['-c:v', 'libx264', '-preset', 'medium', '-crf', '17']

cmd = ['ffmpeg', '-y', '-i', master_path] + v_flags + [
    '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '256k',
    '-movflags', '+faststart', delivery_path]
subprocess.run(cmd, check=True)

print('Entrega generada:', delivery_path)

## 12. Descargar resultados

In [ ]:
from google.colab import files
import os

print('Tamaño delivery (MB):', round(os.path.getsize(delivery_path)/1e6, 1))
print('Tamaño master (MB):  ', round(os.path.getsize(master_path)/1e6, 1))

files.download(delivery_path)
# files.download(master_path)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(master_path, '/content/drive/MyDrive/master_1080p_bt709.mov')

## Notas y límites honestos

- Los flags `--video_backend`/`--use_10bit`/`--chunk_size`/`--cache_dit`/`--cache_vae` se activan **solo si el `--help` instalado los confirma** (celda 1b lo imprime). Si una futura versión del wrapper les cambia el nombre otra vez, aquí no se rompe en silencio — simplemente no se activan y sigue funcionando con el comportamiento base.
- La normalización VFR→CFR del paso 4 es una decisión deliberada, no un intento fallido de preservar VFR: con el motor que usa este notebook, la información de timestamps variables se pierde de todas formas dentro de SeedVR2 (usa OpenCV internamente). Si necesitas preservación exacta de timestamps, este no es el pipeline correcto — haría falta reconstruir con los timestamps originales fuera de este flujo.
- `PER_FRAME_GB`/`MODEL_BASE_GB` para el cálculo de batch por VRAM son estimaciones aproximadas, no medidas exhaustivas — sirven como techo de seguridad razonable, no como garantía absoluta contra OOM.
- La detección de cortes de plano usa el filtro `scene` de ffmpeg con un umbral fijo (`0.4`) — puede fallar en transiciones graduales (fundidos) o sobre-detectar en escenas con mucho movimiento de cámara.
- Sigue sin integrarse FlashVSR/PS-SR/DGAF-VSR/InstantViR/STCDiT ni un benchmark automático entre motores — decisión de alcance, no técnica.